In [1]:
using DelimitedFiles
using CairoMakie
using GLMakie
using LinearAlgebra
using Random
using CSV
using DataFrames
using Serialization

In [2]:
Randomisation = true

save_h_matrix = false
read_h_matrix = false
z_boundary_conditions = false  # if true z_boundary_conditions = true: the periodic boundary conditions are switched on. 

visulize_sim_box = true

copy_size::Int64 = 27 # 27
nx::Int64 = 8     #40#copy_size #24, unit of unit cell length
ny::Int64 = nx    #40#copy_size #24
nz::Int64 = 5     #10#copy_size #15

Interaction_radius_cutoff = true

interaction_cut_off_radius = 0.5 * 1   # interaction radius cut-off in units of number of unit cells 

include("./introduction.jl")
include("./ir_spectra.jl")
include("./ir_spectra_h_matrix.jl")
include("./ir_spectra_centre.jl")
if interaction_cut_off_radius == true
     print("cut-off radius = ", interaction_cut_off_radius * a0_CO, "\n")
end
########################## Read the measured FTIR data ##########################
file_path_p = joinpath("FTIR measured spectra", "FTIR_res0p2cm15K_1p5mm_60kHz_xyzT_38p4_48p5_13p0_140_C12O16_p_pol.txt")
file_p      = open(file_path_p)
header_p    = split(strip(readline(file_p)), '\t')
data_p      = readdlm(file_p, '\t', Float64)   # Read the rest of the file as a matrix of Float64
v_p = data_p[:, 1]
A_p = data_p[:, 2]
 
file_path_s = joinpath("FTIR measured spectra", "FTIR_res0p2cm15K_1p5mm_60kHz_xyzT_38p4_48p5_13p0_140_C12O16_s_pol.txt")
file_s      = open(file_path_s)
header_s    = split(strip(readline(file_s)), '\t')
data_s      = readdlm(file_s, '\t', Float64)   # Read the rest of the file as a matrix of Float64
v_s = data_s[:, 1]
A_s = data_s[:, 2];

In [3]:
ν0 = 2136.97 # cm-1 #2050.0
νk::Vector{Float64} = collect(ν0- 1*range :step:ν0 + 1*range)
nmols_ml = 4*nx*ny*nz

if Randomisation == true    
    Random.seed!(1234)
    num = sign.(rand(nmols_ml) .- 0.5)
    eu_unit_vector = num .* eu # randomised unit vector
else
    eu_unit_vector = eu
end

1280-element Vector{Vector{Float64}}:
 [1.0, 1.0, 1.0]
 [-1.0, 1.0, 1.0]
 [-1.0, 1.0, -1.0]
 [1.0, 1.0, -1.0]
 [1.0, 1.0, 1.0]
 [1.0, -1.0, -1.0]
 [-1.0, 1.0, -1.0]
 [-1.0, -1.0, 1.0]
 [1.0, 1.0, 1.0]
 [1.0, -1.0, -1.0]
 ⋮
 [1.0, 1.0, -1.0]
 [1.0, 1.0, 1.0]
 [-1.0, 1.0, 1.0]
 [1.0, -1.0, 1.0]
 [-1.0, -1.0, 1.0]
 [-1.0, -1.0, -1.0]
 [-1.0, 1.0, 1.0]
 [-1.0, 1.0, -1.0]
 [1.0, 1.0, -1.0]

In [4]:
# @time ipda, isda, ip, is = ir_spectra(νk, eu_unit_vector, com_ol, Δν)

# fig = Figure(size=(900, 600))

# ax = Axis(fig[1,1], xlabel = L"Frequency/cm$^{-1}$", ylabel = L"Intensity/mOD $ $", xgridvisible = false, ygridvisible = false)
# ax.xlabelsize, ax.ylabelsize  = 24, 24
# ax.xticklabelfont, ax.yticklabelfont = "Times New Roman", "Times New Roman"
# ax.xticklabelsize, ax.yticklabelsize = 20, 20

# conversion_to_mOD = 1000

# lines!(ax, v_p, A_p*conversion_to_mOD, color=:red, label = L"p-pol measured$ $")
# lines!(ax, v_s, A_s*conversion_to_mOD, color=:blue, label = L"s-pol measured$ $")

# Normalisation_exciton = maximum(ipda*conversion_to_mOD)/maximum(A_p*conversion_to_mOD)
# lines!(ax, νk, ipda/Normalisation_exciton*conversion_to_mOD, color=:green, label = L"p-pol modelled$ $")
# lines!(ax, νk, isda/Normalisation_exciton*conversion_to_mOD, color=:orange, label = L"s-pol modelled$ $")
# #lines!(ax, νk, ipda_α/Normalisation_exciton*conversion_to_mOD, color=:black, label = L"p-pol ipda_α$ $")
# #lines!(ax, νk, isda_α/Normalisation_exciton*conversion_to_mOD, color=:purple, label = L"s-pol isda_α$ $")

# axislegend(ax, labelsize = 18, position=:lt)
# DataInspector(fig)
# display(fig)

In [5]:
@time ipda_h_matrix, isda_h_matrix, ip_h_matrix, is_h_matrix, h_matrix = ir_spectra_h_matrix(νk, eu_unit_vector, com_ol, Δν)

Matrix{Float64}
(1280, 1280)
2
1280
  1.821092 seconds (70.99 M allocations: 2.134 GiB, 4.90% gc time, 39.21% compilation time)


([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [2135.752537594414 0.8750036072813138 … 0.0020928432532406134 0.010897218318597678; 0.8750036072813138 2136.344983199067 … 0.00964898585515236 0.006991181368104404; … ; 0.0020928432532406134 0.00964898585515236 … 2135.9388123842923 0.8750036072813138; 0.010897218318597678 0.006991181368104404 … 0.8750036072813138 2137.5080577477984])

In [6]:
h_matrix_path = joinpath("h-matrix", "h-matrix_" *string(nx)*"x"*string(ny)*"x"*string(nz)*"_"*"z_bound_"*string(z_boundary_conditions)*".bin")
# save h matrix as binary file
if save_h_matrix == true
    # save txt file in folder of force matrix # writedlm(h_matrix_path, h_matrix)
    # Save to a binary file
    open(h_matrix_path, "w") do io
        serialize(io, h_matrix)
        println("Matrix saved in binary format as 'matrix.bin'")
    end
end

In [7]:
# Load from binary file
if read_h_matrix == true
    h_matrix_large = open(h_matrix_path, "r") do io # takes about 1 min after restart
        deserialize(io)
    end
    # println("Matrix loaded: ", h_matrix_large)    
    # get the diagonal elements for the different energies
    h_matrix_large_energies = diag(h_matrix_large)
    fig = Figure(size=(900, 600))
    ax = Axis(fig[1,1], xlabel = L"Frequency/cm$^{-1}$", ylabel = L"Intensity/mOD $ $", xgridvisible = false, ygridvisible = false)
    lines!(ax, h_matrix_large_energies, color=:red, label = L"energies")
    axislegend(ax, labelsize = 18, position=:lt)
    DataInspector(fig)
    display(fig)
end

In [8]:
small_box_size_fraction = 3
# get the index of each molecules and define the middle position of cuboid 
nx_centre::Int64 = nx÷(small_box_size_fraction+0.00000)   # get subset of alpha-CO which is smaller than small_box_size_fraction-times the size of the total simulation box (use modulo ÷ to get unit cell below exactly small_box_size_fraction the size)
ny_centre::Int64 = ny÷(small_box_size_fraction+0.00000) 
nz_centre::Int64 = nz

println(nx_centre,' ', ny_centre,' ', nz_centre)
nmols_centre = 4*nx_centre*ny_centre*nz_centre

2 2 5


80

In [9]:
max_x_centre = maximum(com_ol[:, 1]) * nx_centre / nx
max_y_centre = maximum(com_ol[:, 2]) * ny_centre / ny
max_z_centre = maximum(com_ol[:, 3]) * nz_centre / nz
print(max_x_centre, ' ', max_y_centre, ' ', max_z_centre)

In [10]:
# indexing of molecules in the centre of the simulation box:
com_ol_centre         = zeros(Float64, nmols_centre, 3) 
eu_unit_vector_centre = Vector{Vector{Float64}}(undef, nmols_centre)
large_h_matrix_sum    = zeros(Float64, nmols_centre)
position_z_centre = []

count = 1
for row_i in 1:size(com_ol, 1)
    if com_ol[row_i, 1] <= max_x_centre && com_ol[row_i, 2] <= max_y_centre && com_ol[row_i, 3] <= max_z_centre
        com_ol_centre[count, 1] = com_ol[row_i, 1]
        com_ol_centre[count, 2] = com_ol[row_i, 2]
        com_ol_centre[count, 3] = com_ol[row_i, 3]
        eu_unit_vector_centre[count] = eu_unit_vector[row_i]
        large_h_matrix_sum[count] = h_matrix[row_i, row_i]
        count += 1
        if nz % 2 == 0
            if com_ol[row_i, 1] == 0.0 && com_ol[row_i, 2] == 0.0 && com_ol[row_i, 3] == (max_z_centre*2+1)/4 # get most central molecule in z-direction
                position_z_centre = com_ol[row_i, :]
            end
        else
            if com_ol[row_i, 1] == 0.0 && com_ol[row_i, 2] == 0.0 && com_ol[row_i, 3] == (max_z_centre*2-1)/4 # get most central molecule in z-direction
                position_z_centre = com_ol[row_i, :]
            end
        end
        
    end
end
position_z_centre = position_z_centre * a0_CO .+ a0_CO/4  # m, unit conversion, +a0_CO/2 to get the centre of the interaction radius without cutting off unit-cell

1.875 1.875 4.5

3-element Vector{Float64}:
 1.41e-10
 1.41e-10
 1.269e-9

In [11]:
if visulize_sim_box == true  # visualize the simulation box:

    X = com_ol[:, 1]
    Y = com_ol[:, 2]
    Z = com_ol[:, 3]

    U = [vec[1] for vec in eu_unit_vector]
    V = [vec[2] for vec in eu_unit_vector]
    W = [vec[3] for vec in eu_unit_vector]

    index_vector_of_centre_molecules = [1:nmols_centre;]

    X_centre = com_ol_centre[:, 1]
    Y_centre = com_ol_centre[:, 2]
    Z_centre = com_ol_centre[:, 3]

    U_centre = [vec[1] for vec in eu_unit_vector_centre]
    V_centre = [vec[2] for vec in eu_unit_vector_centre]
    W_centre = [vec[3] for vec in eu_unit_vector_centre]

    # Create a figure and 3D axis
    fig = Figure()
    ax = Axis3(fig[1, 1], xlabel = "X Coordinate", ylabel = "Y Coordinate", zlabel = "Z Coordinate")

    arrows!(ax,  vec(X), vec(Y), vec(Z), 0.2 .* vec(U), 0.2 .* vec(V), 0.2 .* vec(W), arrowsize=0.1, color=:red)
    scatter!(ax, vec(X_centre), vec(Y_centre), vec(Z_centre), markersize=40)
    arrows!(ax,  vec(X_centre), vec(Y_centre), vec(Z_centre), 0.2 .* vec(U_centre), 0.2 .* vec(V_centre), 0.2 .* vec(W_centre), arrowsize=0.15, color=:blue)

    ax.title = "3D Grid of Molecules with Orientation Arrows"
    # Display the figure
    display(fig)
end

GLMakie.Screen(...)

In [12]:
using CairoMakie, GeometryBasics

if visulize_sim_box == true && Interaction_radius_cutoff == true
    # visualize the simulation box:

    X = com_ol[:, 1] .* a0_CO
    Y = com_ol[:, 2] .* a0_CO
    Z = com_ol[:, 3] .* a0_CO

    U = [vec[1] for vec in eu_unit_vector] .* a0_CO
    V = [vec[2] for vec in eu_unit_vector] .* a0_CO
    W = [vec[3] for vec in eu_unit_vector] .* a0_CO

    index_vector_of_centre_molecules = [1:nmols_centre;]

    X_centre = com_ol_centre[:, 1] .* a0_CO
    Y_centre = com_ol_centre[:, 2] .* a0_CO
    Z_centre = com_ol_centre[:, 3] .* a0_CO

    U_centre = [vec[1] for vec in eu_unit_vector_centre] .* a0_CO
    V_centre = [vec[2] for vec in eu_unit_vector_centre] .* a0_CO
    W_centre = [vec[3] for vec in eu_unit_vector_centre] .* a0_CO

    # Define sphere properties
    center = Point3f0(position_z_centre[1]*1e10, position_z_centre[2]*1e10, position_z_centre[3]*1e10)  # Center of the sphere
    r_0::Float32 = interaction_cut_off_radius * a0_CO *1e10  # Radius of the sphere
    sphere = HyperSphere(center, r_0)  # Define a sphere

    # Create a figure and 3D axis
    fig = Figure()
    ax = Axis3(fig[1, 1], xlabel = "X Coordinate / Å", ylabel = "Y Coordinate / Å", zlabel = "Z Coordinate / Å")

    arrows!(ax,  vec(X) * 1e10, vec(Y) * 1e10, vec(Z) * 1e10, 0.2 .* vec(U) * 1e10, 0.2 .* vec(V) * 1e10, 0.2 .* vec(W) * 1e10, arrowsize=0.1, color=:red)
    scatter!(ax, vec(X_centre) * 1e10, vec(Y_centre) * 1e10, vec(Z_centre) * 1e10, markersize=40)
    arrows!(ax,  vec(X_centre) * 1e10, vec(Y_centre) * 1e10, vec(Z_centre) * 1e10, 0.2 .* vec(U_centre) * 1e10, 0.2 .* vec(V_centre) * 1e10, 0.2 .* vec(W_centre) * 1e10, arrowsize=0.15, color=:blue)

    mesh!(ax, sphere, color = (:orange, 0.6))  # Blue color with 50% transparency

    ax.title = "3D Grid of Molecules with Orientation Arrows"
    # Display the figure
    display(fig)
end

GLMakie.Screen(...)

In [13]:
Interaction_radius_cutoff = true
#interaction_cut_off_radius = 1   # m interaction radius cut-off in # of unit cells

# read the force matrix for the maximum size and do eigen(h) operation of small simulation box with entries of the large h matrix on the diagonal
@time ipda, isda, ip, is, eigenvecs = ir_spectra_centre(νk, eu_unit_vector_centre,  com_ol_centre, Δν, nmols_centre, large_h_matrix_sum)

80(80, 80)
  0.701824 seconds (3.33 M allocations: 113.721 MiB, 3.46% gc time, 97.19% compilation time)


([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0])

In [14]:
α = 0*degrees 

ipda_α = (cos(α))^2 .* ipda + (sin(α))^2 .* isda 
isda_α = (cos(α))^2 .* isda + (sin(α))^2 .* ipda;

In [15]:
# CairoMakie.activate!()
# GLMakie.activate!()
fig = Figure(size=(900, 600))

ax = Axis(fig[1,1], xlabel = L"Frequency/cm$^{-1}$", ylabel = L"Intensity/mOD $ $", xgridvisible = false, ygridvisible = false)
ax.xlabelsize, ax.ylabelsize  = 24, 24
ax.xticklabelfont, ax.yticklabelfont = "Times New Roman", "Times New Roman"
ax.xticklabelsize, ax.yticklabelsize = 20, 20

conversion_to_mOD = 1000

lines!(ax, v_p, A_p*conversion_to_mOD, color=:red, label = L"p-pol measured$ $")
lines!(ax, v_s, A_s*conversion_to_mOD, color=:blue, label = L"s-pol measured$ $")

Normalisation_exciton = maximum(ipda*conversion_to_mOD)/maximum(A_p*conversion_to_mOD)
lines!(ax, νk, ipda/Normalisation_exciton*conversion_to_mOD, color=:green, label = L"p-pol modelled$ $")
lines!(ax, νk, isda/Normalisation_exciton*conversion_to_mOD, color=:orange, label = L"s-pol modelled$ $")

#lines!(ax, νk, ipda_α/Normalisation_exciton*conversion_to_mOD, color=:black, label = L"p-pol ipda_α$ $")
#lines!(ax, νk, isda_α/Normalisation_exciton*conversion_to_mOD, color=:purple, label = L"s-pol isda_α$ $")

axislegend(ax, labelsize = 18, position=:lt)
DataInspector(fig)
display(fig)

GLMakie.Screen(...)